# Dopamine PINN: training remedies for the multi-scale failures

Companion to `dopamine_PINN_extensions.ipynb`. There the PINN failed in two settings
where the reference estimator succeeded:

* **E3b**: Michaelis-Menten uptake over a 500 ms record with a 10 µM peak (PINN: D -23%,
  Km +120-150%, consistent over 3 seeds; the PDE term was effectively ~100x weaker
  after time rescaling).
* **E5**: diffusion around cellular obstacles with D_free = 0.763 µm²/ms (PINN: D +62-70%,
  k -28%; a pinned-truth run had a *higher* loss, i.e. the objective favoured the wrong D
  while the ~0.2 ms release transient was under-resolved).

Three standard remedies are tested, alone and combined, on both cases:

| Variant | Remedy |
|---|---|
| `base` | the paper's setting (fixed weights 3000 / 1 / 100 / 5, uniform collocation) |
| `anneal` | **adaptive loss weights**, learning-rate annealing of Wang, Teng & Perdikaris (2021): every 100 Adam steps, λ̂_j = max\|∇(λ_r L_r)\| / mean\|∇L_j\| for the IC, BC and data terms, λ_j ← (1-α)λ_j + αλ̂_j, kept within 100x of the paper weights (`ANNEAL_CAP`) |
| `rad` | **residual-based adaptive distribution** (RAD, Wu et al. 2023): every 2000 Adam steps the interior points are redrawn from a large pool with probability ∝ \|r\|^k/mean\|r\|^k + c |
| `ff` | **Fourier-feature input** (Tancik et al. 2020): the network sees [x, y, t, sin(2πBz), cos(2πBz)] with z the coordinates scaled to [0,1] and B ~ N(0, s²) fixed |
| `all` | all three |

Each variant runs on 2 network seeds per case. **Pinned-truth diagnostic**: for `base` and
`all`, a run with the physical parameters fixed at the truth; its loss, evaluated with the
paper's fixed weights, is compared with the learned run's. If the truth now has the lower
loss, the remedy fixed the objective, not just the optimizer path.
Each run also reports the network's error against the FD solution at several times, so a
remedy that helps the parameters for the wrong reason is visible.

Optional `RUN_E1_SUBSET`: the chosen variant on the first 5 of the 20 E1 realizations
(seeds 7001-7005), to compare with the baseline values of E1.

| Toggle | Runs | A100 time (est.) |
|---|---|---|
| `RUN_E3B` | 5 variants x 2 seeds + 2 pinned = 12 | ~40 min |
| `RUN_E5`  | 12 | ~40 min |
| `RUN_E1_SUBSET` | 5 | ~15 min |

Results are saved after every run (to Google Drive with `USE_DRIVE = True`), and a rerun
skips finished runs. The last cell zips and downloads everything.

## 0. Install dependencies (same pins as the other notebooks)

In [ ]:
import os
from pathlib import Path

SENTINEL = Path('/tmp/jax_pinn_deps_ready_v2')
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB and not SENTINEL.exists():
    # Pinned, mutually compatible versions (an unpinned flax/jax upgrade breaks
    # `from flax import nnx`).
    rc = os.system('pip -q install '
                   '"jax[cuda12]==0.10.2" '
                   '"flax==0.12.5" '
                   '"optax>=0.2.3" '
                   '"jaxopt>=0.8" '
                   'scipy matplotlib')
    if rc != 0:
        raise RuntimeError('pip install failed - see the output above.')
    SENTINEL.touch()
    print('Install complete. Restarting kernel; then Runtime -> Run all again.')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies ready.')

In [ ]:
import os, time, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc, chi2
from scipy.optimize import least_squares

import jax
import jax.numpy as jnp
from jax import lax
from flax import nnx
import optax
from jaxopt import LBFGS

jax.config.update('jax_enable_x64', True)
print(f'JAX {jax.__version__}, devices: {jax.devices()}')

## 1. Toggles and parameters

In [ ]:
# ---- Toggles -----------------------------------------------------------------
QUICK         = False   # tiny smoke test (CPU is fine); not paper-quality
USE_DRIVE     = True    # save to Google Drive so a disconnect does not lose runs
RUN_E3B       = True    # Michaelis-Menten, 500 ms record                    (~40 min A100)
RUN_E5        = True    # obstacles                                          (~40 min A100)
RUN_E1_SUBSET = False   # chosen variant on E1 realizations 7001-7005       (~15 min A100)
E1_VARIANT    = 'rad'   # set after looking at the E3b / E5 results

VARIANTS = {               # remedy switches per variant
    'base':   dict(anneal=False, rad=False, ff=False),
    'anneal': dict(anneal=True,  rad=False, ff=False),
    'rad':    dict(anneal=False, rad=True,  ff=False),
    'ff':     dict(anneal=False, rad=False, ff=True),
    'all':    dict(anneal=True,  rad=True,  ff=True),
}
NET_SEEDS = [1234, 1235]
PINNED_VARIANTS = ['base', 'all']

# ---- Remedy settings ----------------------------------------------------------
ANNEAL_EVERY = 100       # Adam steps between weight updates (Wang et al. 2021)
ANNEAL_ALPHA = 0.9       # moving-average factor of Wang et al. 2021
ANNEAL_CAP   = 100.0     # weights kept within [w/CAP, w*CAP] of the paper weights; None = uncapped.
                         # Uncapped, the max/mean ratio drove the BC weight from 100 to ~1e8
                         # and the data weight from 5 to ~7e5 within 400 steps in a smoke test,
                         # which makes the network fit the noise.
RAD_EVERY    = 2_000     # Adam steps between resamplings
RAD_POOL     = 4         # pool size = RAD_POOL x N_DOMAIN
RAD_K, RAD_C = 1.0, 1.0  # p ~ |r|^k / mean(|r|^k) + c (Wu et al. 2023 defaults)
FF_M         = 32        # number of Fourier frequencies
FF_SIGMA     = 3.0       # SD of B (cycles per unit of scaled coordinate)

# ---- Physics (as in the paper) ---------------------------------------------------
D_TRUE, K_TRUE = 0.32, 0.020       # um^2/ms, 1/ms
L, T, SIGMA, C0 = 5.0, 50.0, 0.5, 1.0
D_INIT, K_INIT = 0.30, 0.016       # offset initial guesses (-6.25%, -20%)
NOISE_PCT = 2.0
T_OBS = (0.5, 0.9 * T)
N_OBS = 400

# ---- Network and training (final inverse setting of the paper) -------------------
LAYERS      = [3] + [64] * 4 + [1]
ADAM_ITERS  = 20_000
LBFGS_ITERS = 2_000
LR          = 1e-3
N_CHUNK     = 50
N_DOMAIN    = 30_000
N_INITIAL   = 400
N_BOUNDARY  = 2_000
W_RES, W_IC, W_BC, W_D = 3000.0, 1.0, 100.0, 5.0
SEED = 1234

# ---- FD reference solver ---------------------------------------------------------
NX_FD    = 81
DT_STORE = 0.1

if QUICK:
    ADAM_ITERS, LBFGS_ITERS, N_CHUNK = 400, 50, 50
    N_DOMAIN, N_INITIAL, N_BOUNDARY = 1_200, 200, 400
    N_OBS = 200
    NX_FD, DT_STORE = 41, 0.25
    ANNEAL_EVERY, RAD_EVERY = 100, 200
    NET_SEEDS = [1234]
    print('[QUICK MODE] reduced settings - not paper-quality.')
assert ADAM_ITERS % N_CHUNK == 0 and ANNEAL_EVERY % N_CHUNK == 0 and RAD_EVERY % N_CHUNK == 0

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/dopamine_PINN_remedies')
else:
    OUT_DIR = Path('remedies_out')
if QUICK:
    OUT_DIR = OUT_DIR / 'quick'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Results -> {OUT_DIR.resolve()}')

## 2. FD reference solver (JAX) and reference estimator

Node-centred flux form on an `nx x nx` grid over $[-L/2, L/2]^2$:
$\partial_t C_{ij} = [F^x_{i+1/2,j} - F^x_{i-1/2,j}]/(w_i\,\Delta x^2) + [\ldots]^y - s(C_{ij})$,
with face flux $F^x_{i+1/2,j} = \bar D\,(C_{i+1,j} - C_{ij})$, $\bar D$ the mean of the two
nodal values, and $w_i = 1/2$ on the walls (half control volume, zero wall flux). For
constant $D$ this is the mirror-Neumann scheme of the main notebook. An obstacle mask
sets the flux through every face that touches a solid node to zero. The time step is
fixed by a static cap `D_cap` on $D$ so the solver can be differentiated with respect to
$D$ (forward mode), and the reference estimator gets exact Jacobians.

In [ ]:
def gaussian_ic(nx, amp=C0):
    x = np.linspace(-L / 2, L / 2, nx)
    X, Y = np.meshgrid(x, x, indexing='ij')
    return amp * np.exp(-(X**2 + Y**2) / (2 * SIGMA**2))


def make_fd_solver(nx=None, D_cap=0.64, mask=None, T_end=None, dt_store=None):
    """Returns solve(D_nodes, sink, C_init) -> frames (n_frames + 1, nx, nx).

    Frames are stored at t = j * dt_store up to T_end (defaults DT_STORE, T).
    D_nodes is a scalar or (nx, nx) array with D <= D_cap; sink(C) is the uptake
    rate (e.g. k * C).
    """
    nx = NX_FD if nx is None else nx
    T_end = T if T_end is None else T_end
    dt_store = DT_STORE if dt_store is None else dt_store
    dx = L / (nx - 1)
    dt_max = 0.95 * dx**2 / (4.0 * D_cap)
    stride = int(np.ceil(dt_store / dt_max))
    dt = dt_store / stride
    n_frames = int(round(T_end / dt_store))
    m = np.ones((nx, nx)) if mask is None else mask.astype(float)
    mfx = jnp.asarray(m[1:, :] * m[:-1, :])
    mfy = jnp.asarray(m[:, 1:] * m[:, :-1])
    w = np.ones(nx); w[0] = w[-1] = 0.5
    ivx = jnp.asarray(1.0 / (dx**2 * w))[:, None]
    ivy = jnp.asarray(1.0 / (dx**2 * w))[None, :]
    mj = jnp.asarray(m)

    def solve(D_nodes, sink, C_init):
        Dn = jnp.broadcast_to(jnp.asarray(D_nodes, jnp.float64), (nx, nx))
        Dfx = 0.5 * (Dn[1:, :] + Dn[:-1, :]) * mfx
        Dfy = 0.5 * (Dn[:, 1:] + Dn[:, :-1]) * mfy

        def step(_, C):
            Fx = Dfx * (C[1:, :] - C[:-1, :])
            Fy = Dfy * (C[:, 1:] - C[:, :-1])
            div = ((jnp.pad(Fx, ((0, 1), (0, 0))) - jnp.pad(Fx, ((1, 0), (0, 0)))) * ivx
                   + (jnp.pad(Fy, ((0, 0), (0, 1))) - jnp.pad(Fy, ((0, 0), (1, 0)))) * ivy)
            return C + dt * (div - sink(C)) * mj

        def frame(C, _):
            C = lax.fori_loop(0, stride, step, C)
            return C, C

        C_init = jnp.asarray(C_init) * mj
        _, F = lax.scan(frame, C_init, None, length=n_frames)
        return jnp.concatenate([C_init[None], F])

    solve.nx, solve.dx, solve.dt, solve.mask = nx, dx, dt, m
    solve.dt_store, solve.T_end = dt_store, T_end
    return solve


def interp_frames(frames, dx, xq, yq, tq, dt_store=None):
    """Trilinear interpolation of stored frames at (x, y, t) points (JAX)."""
    nf, nx, _ = frames.shape
    ft = tq / (DT_STORE if dt_store is None else dt_store)
    it = jnp.clip(jnp.floor(ft).astype(int), 0, nf - 2); a = ft - it
    fx = (xq + L / 2) / dx
    ix = jnp.clip(jnp.floor(fx).astype(int), 0, nx - 2); b = fx - ix
    fy = (yq + L / 2) / dx
    iy = jnp.clip(jnp.floor(fy).astype(int), 0, nx - 2); c = fy - iy

    def bil(ti):
        return ((1 - b) * (1 - c) * frames[ti, ix, iy] + b * (1 - c) * frames[ti, ix + 1, iy]
                + (1 - b) * c * frames[ti, ix, iy + 1] + b * c * frames[ti, ix + 1, iy + 1])
    return (1 - a) * bil(it) + a * bil(it + 1)


def ref_fit(predict, theta0, C_obs, sigma, lb, ub):
    """Reference estimator: bounded non-linear least squares through the FD solver.

    predict(theta) -> concentrations at the observation points (JAX, differentiable).
    Returns the estimate, the Gauss-Newton (Laplace) covariance of theta, the residual
    RMS and chi^2 statistics. The Jacobian is exact (forward-mode autodiff).
    """
    f = jax.jit(predict)
    J = jax.jit(jax.jacfwd(predict))
    th0 = np.clip(np.asarray(theta0, float), np.asarray(lb) + 1e-9, np.asarray(ub) - 1e-9)
    r = least_squares(lambda th: np.asarray(f(jnp.asarray(th))) - C_obs, th0,
                      jac=lambda th: np.asarray(J(jnp.asarray(th))),
                      bounds=(lb, ub), x_scale='jac', xtol=1e-10, ftol=1e-12, gtol=1e-10)
    Jm = r.jac
    cov = np.linalg.pinv(Jm.T @ Jm) * sigma**2
    n, p = C_obs.size, len(theta0)
    rss = float(np.sum(r.fun**2))
    return {'theta': r.x, 'cov': cov, 'rms': float(np.sqrt(rss / n)),
            'chi2': rss / sigma**2, 'dof': n - p,
            'p_chi2': float(chi2.sf(rss / sigma**2, n - p)),
            'aic': n * np.log(rss / n) + 2 * p, 'nfev': int(r.nfev),
            'at_bound': bool(np.any(np.isclose(r.x, lb) | np.isclose(r.x, ub)))}


def scattered_obs(frames, solver, seed, n_obs=None, noise_pct=NOISE_PCT, valid=None, t_obs=T_OBS):
    """Uniform (x, y, t) observations with Gaussian noise (draw order as the main notebook).

    valid(x, y) -> bool restricts locations (obstacles); rejected points are redrawn.
    """
    n_obs = N_OBS if n_obs is None else n_obs
    rng = np.random.default_rng(seed)
    if valid is None:
        ox = rng.uniform(-L / 2, L / 2, n_obs); oy = rng.uniform(-L / 2, L / 2, n_obs)
    else:
        ox, oy = np.empty(0), np.empty(0)
        while ox.size < n_obs:
            cx = rng.uniform(-L / 2, L / 2, 4 * n_obs); cy = rng.uniform(-L / 2, L / 2, 4 * n_obs)
            ok = valid(cx, cy)
            ox = np.concatenate([ox, cx[ok]]); oy = np.concatenate([oy, cy[ok]])
        ox, oy = ox[:n_obs], oy[:n_obs]
    ot = rng.uniform(*t_obs, n_obs)
    clean = np.asarray(interp_frames(frames, solver.dx, jnp.asarray(ox), jnp.asarray(oy), jnp.asarray(ot),
                                     solver.dt_store))
    noisy = clean + rng.normal(0.0, noise_pct / 100 * C0, n_obs)
    return {'x_d': ox, 'y_d': oy, 't_d': ot, 'C_d': noisy, 'C_clean': clean}


# ---- check against the numpy solver of the main notebook ------------------------
def fd_reference_np(D=D_TRUE, k=K_TRUE, nx=81, dt_store=0.02):
    dx = L / (nx - 1)
    nt = int(np.ceil(T / (0.95 / (2.0 * D * (2.0 / dx**2))))) + 1
    dt = T / (nt - 1)
    C = gaussian_ic(nx)
    for n in range(1, nt):
        lap = np.zeros_like(C)
        lap[1:-1, :] += (C[2:, :] - 2 * C[1:-1, :] + C[:-2, :]) / dx**2
        lap[0, :] += 2 * (C[1, :] - C[0, :]) / dx**2
        lap[-1, :] += 2 * (C[-2, :] - C[-1, :]) / dx**2
        lap[:, 1:-1] += (C[:, 2:] - 2 * C[:, 1:-1] + C[:, :-2]) / dx**2
        lap[:, 0] += 2 * (C[:, 1] - C[:, 0]) / dx**2
        lap[:, -1] += 2 * (C[:, -2] - C[:, -1]) / dx**2
        C = C + D * dt * lap - k * dt * C
    return C


SOLVE_LIN = make_fd_solver(D_cap=0.64)
t0 = time.time()
FR_LIN = jax.block_until_ready(SOLVE_LIN(D_TRUE, lambda C: K_TRUE * C, gaussian_ic(SOLVE_LIN.nx)))
print(f'JAX FD solve: {time.time() - t0:.1f} s (incl. compile), frames {FR_LIN.shape}')
C_np = fd_reference_np(nx=SOLVE_LIN.nx)
print(f'max |JAX - numpy| at t = T: {np.max(np.abs(np.asarray(FR_LIN[-1]) - C_np)):.2e} '
      f'(peak {C_np.max():.4f}; the two differ only in time step)')

## 3. PINN building blocks

One MLP class for the concentration network (and a smaller one for the $D(x, y)$
network of E4). A loss is assembled from a `physics(p, x, y, t, C, grad, lap)` function
that returns the PDE residual at one point; the trainer is the Adam-then-L-BFGS loop of
the paper. Collocation, initial and boundary points can be restricted to the
extracellular space and include obstacle boundaries (E5).

In [ ]:
class MLP(nnx.Module):
    """tanh MLP, float64 parameters (dtype-homogeneous pytree for jaxopt L-BFGS)."""
    def __init__(self, layers, *, rngs, zero_last=False):
        self.n_layers = len(layers) - 1
        for i in range(self.n_layers):
            last = zero_last and i == self.n_layers - 1
            setattr(self, f'lin_{i}', nnx.Linear(
                layers[i], layers[i + 1],
                kernel_init=(nnx.initializers.zeros_init() if last
                             else nnx.initializers.glorot_normal()),
                bias_init=nnx.initializers.zeros_init(),
                param_dtype=jnp.float64, rngs=rngs))

    def __call__(self, z):
        h = z
        for i in range(self.n_layers - 1):
            h = jnp.tanh(getattr(self, f'lin_{i}')(h))
        return getattr(self, f'lin_{self.n_layers - 1}')(h)


GDEF = nnx.split(MLP(LAYERS, rngs=nnx.Rngs(0)))[0]
D_LAYERS = [2, 32, 32, 1]
GDEF_D = nnx.split(MLP(D_LAYERS, rngs=nnx.Rngs(0), zero_last=True))[0]


FF_B = jnp.asarray(np.random.default_rng(2024).normal(0.0, FF_SIGMA, (FF_M, 3)))
GDEF_FF = nnx.split(MLP([3 + 2 * FF_M] + LAYERS[1:], rngs=nnx.Rngs(0)))[0]


def init_net(seed, ff=False):
    layers = [3 + 2 * FF_M] + LAYERS[1:] if ff else LAYERS
    return nnx.split(MLP(layers, rngs=nnx.Rngs(seed)))[1]


def net_fn(p, t_scale=1.0, arch=None):
    """Scalar C(x, y, t). t_scale < 1 maps a long record onto the paper's t range [0, 50].

    arch = {'ff': True, 'T_end': T_end} adds Fourier features of the coordinates scaled to
    [0, 1]: input [x, y, t * t_scale, sin(2 pi B z), cos(2 pi B z)], z = (x / L, y / L,
    t / T_end), B fixed (FF_B).
    """
    sc = jnp.array([1.0, 1.0, 1.0]).at[2].set(t_scale)
    if not arch or not arch.get('ff'):
        net = nnx.merge(GDEF, p['net'])
        return lambda z: net(z * sc)[0]
    net = nnx.merge(GDEF_FF, p['net'])
    nrm = jnp.array([L, L, arch['T_end']])
    def f(z):
        zz = 2 * jnp.pi * (FF_B @ (z / nrm))
        return net(jnp.concatenate([z * sc, jnp.sin(zz), jnp.cos(zz)]))[0]
    return f


EX, EY = jnp.array([1.0, 0.0, 0.0]), jnp.array([0.0, 1.0, 0.0])


def point_derivs(f, x, y, t):
    """C, grad (C_x, C_y, C_t) and the Laplacian C_xx + C_yy at one point."""
    z = jnp.stack([x, y, t])
    g = jax.grad(f)
    gz, hx = jax.jvp(g, (z,), (EX,))
    _, hy = jax.jvp(g, (z,), (EY,))
    return f(z), gz, hx[0] + hy[1]


# ---- physics (PDE residuals) ------------------------------------------------------
def phys_linear(p, x, y, t, C, g, lap):
    return g[2] - jnp.exp(p['logD']) * lap + jnp.exp(p['logk']) * C


def phys_mm(p, x, y, t, C, g, lap):
    # |C| keeps the denominator positive if the network dips below zero early on
    Vmax, Km = jnp.exp(p['logVmax']), jnp.exp(p['logKm'])
    return g[2] - jnp.exp(p['logD']) * lap + Vmax * C / (Km + jnp.abs(C))


D_REF_FIELD = D_INIT    # D(x, y) = D_REF_FIELD * exp(dnet(x, y)); dnet starts at 0


def d_field(p):
    dn = nnx.merge(GDEF_D, p['dnet'])
    return lambda xy: D_REF_FIELD * jnp.exp(dn(xy)[0])


def phys_dfield(p, x, y, t, C, g, lap):
    Df = d_field(p)
    xy = jnp.stack([x, y])
    D, gD = Df(xy), jax.grad(Df)(xy)
    return g[2] - (D * lap + gD[0] * g[0] + gD[1] * g[1]) + jnp.exp(p['logk']) * C


def build_loss(physics, w_d=W_D, extra=None, w_res=None, pin=None, arch=None):
    """Total loss W_RES L_r + W_IC L_i + W_BC L_b + w_d L_d (+ extra(p, b))."""
    def components(p, b):
        f = net_fn(p, b.get('t_scale', 1.0), arch)
        res = jax.vmap(lambda x, y, t: physics(p, x, y, t, *point_derivs(f, x, y, t)))(
            b['x_r'], b['y_r'], b['t_r'])
        C_ic = jax.vmap(lambda x, y: f(jnp.stack([x, y, 0.0])))(b['x_i'], b['y_i'])
        dn = jax.vmap(lambda x, y, t, nx, ny:
                      jnp.dot(jax.grad(f)(jnp.stack([x, y, t]))[:2], jnp.stack([nx, ny])))(
            b['x_b'], b['y_b'], b['t_b'], b['nx_b'], b['ny_b'])
        C_d = jax.vmap(lambda x, y, t: f(jnp.stack([x, y, t])))(b['x_d'], b['y_d'], b['t_d'])
        return (jnp.mean(res**2), jnp.mean((C_ic - b['C_i'])**2),
                jnp.mean(dn**2), jnp.mean((C_d - b['C_d'])**2))

    w_r = W_RES if w_res is None else w_res
    full = (lambda p: p) if pin is None else (lambda p: {**p, **pin})   # pin: fixed entries, e.g. {'logD': ...}

    def loss(p, b):
        L_r, L_i, L_b, L_d = components(full(p), b)
        tot = w_r * L_r + W_IC * L_i + W_BC * L_b + w_d * L_d
        return tot + (extra(p, b) if extra is not None else 0.0)

    loss.components = lambda p, b: components(full(p), b)
    loss.w_res, loss.arch = w_r, arch
    return loss


def train(params, loss, batch, label='', report=None):
    """Adam (ADAM_ITERS, lr LR) then L-BFGS (LBFGS_ITERS, tol 1e-9), as in the paper."""
    opt = optax.adam(LR)
    state = opt.init(params)

    @jax.jit
    def chunk(p, s, b):
        def body(_, carry):
            p, s, _l = carry
            l, g = jax.value_and_grad(loss)(p, b)
            u, s = opt.update(g, s, p)
            return optax.apply_updates(p, u), s, l
        return lax.fori_loop(0, N_CHUNK, body, (p, s, jnp.zeros((), jnp.float64)))

    t0 = time.time()
    n_chunks = ADAM_ITERS // N_CHUNK
    for c in range(n_chunks):
        params, state, l = chunk(params, state, batch)
        if c == 0 or (c + 1) % 80 == 0 or c == n_chunks - 1:
            extra = report(params) if report else ''
            print(f'    [{label}] step {(c + 1) * N_CHUNK:>5d}  loss {float(l):.4e}  {extra}  '
                  f'{time.time() - t0:.0f}s')
    params = LBFGS(fun=loss, maxiter=LBFGS_ITERS, tol=1e-9).run(params, b=batch).params
    print(f'    [{label}] after L-BFGS: loss {float(loss(params, batch)):.4e}  '
          f'{report(params) if report else ""}  total {time.time() - t0:.0f}s')
    return params


# ---- collocation ------------------------------------------------------------------
def sample_points(n_domain=None, n_initial=None, n_boundary=None, seed=SEED,
                  obstacles=None, n_obstacle_bc=0, T_end=T, amp=C0, t_power=1.0, t_c=None):
    """LHS interior points, uniform IC points, wall points (+ obstacle-boundary points).

    obstacles: list of (xc, yc, R) discs; interior and IC points inside them are
    rejected, and n_obstacle_bc boundary points are spread over the circles in
    proportion to circumference, with the radial unit normal. For a long record
    (T_end > T) t_power > 1 puts more interior points at early times (t = T_end u^p).
    t_c given: interior and boundary times are uniform in s = log(1 + t / t_c) instead.
    """
    n_domain = N_DOMAIN if n_domain is None else n_domain
    n_initial = N_INITIAL if n_initial is None else n_initial
    n_boundary = N_BOUNDARY if n_boundary is None else n_boundary
    rng = np.random.default_rng(seed)
    outside = (lambda x, y: np.ones_like(x, bool)) if not obstacles else (
        lambda x, y: np.all([(x - a)**2 + (y - b)**2 > R**2 for a, b, R in obstacles], axis=0))

    over = 1.0 if not obstacles else 1.6
    u = qmc.LatinHypercube(d=3, seed=seed).random(int(over * n_domain))
    x_r, y_r, t_r = -L / 2 + L * u[:, 0], -L / 2 + L * u[:, 1], T_end * u[:, 2]**t_power
    if t_c is not None:
        t_r = t_c * np.expm1(u[:, 2] * np.log1p(T_end / t_c))
    keep = outside(x_r, y_r)
    x_r, y_r, t_r = x_r[keep][:n_domain], y_r[keep][:n_domain], t_r[keep][:n_domain]

    x_i, y_i = np.empty(0), np.empty(0)
    while x_i.size < n_initial:
        cx, cy = rng.uniform(-L / 2, L / 2, 2 * n_initial), rng.uniform(-L / 2, L / 2, 2 * n_initial)
        ok = outside(cx, cy)
        x_i, y_i = np.concatenate([x_i, cx[ok]]), np.concatenate([y_i, cy[ok]])
    x_i, y_i = x_i[:n_initial], y_i[:n_initial]
    C_i = amp * np.exp(-(x_i**2 + y_i**2) / (2 * SIGMA**2))

    pe = n_boundary // 4
    s = lambda: rng.uniform(-L / 2, L / 2, pe)
    tt = ((lambda n: rng.uniform(0, T_end, n)) if t_c is None else
          (lambda n: t_c * np.expm1(rng.uniform(0, 1, n) * np.log1p(T_end / t_c))))
    edges = [(np.full(pe, -L / 2), s(), tt(pe), -np.ones(pe), np.zeros(pe)),
             (np.full(pe, L / 2), s(), tt(pe), np.ones(pe), np.zeros(pe)),
             (s(), np.full(pe, -L / 2), tt(pe), np.zeros(pe), -np.ones(pe)),
             (s(), np.full(pe, L / 2), tt(pe), np.zeros(pe), np.ones(pe))]
    if obstacles and n_obstacle_bc:
        circ = np.array([R for _, _, R in obstacles])
        counts = np.round(n_obstacle_bc * circ / circ.sum()).astype(int)
        for (a, b_, R), n in zip(obstacles, counts):
            th = rng.uniform(0, 2 * np.pi, n)
            edges.append((a + R * np.cos(th), b_ + R * np.sin(th), tt(n), np.cos(th), np.sin(th)))
    cat = lambda i: np.concatenate([e[i] for e in edges])
    return {'x_r': x_r, 'y_r': y_r, 't_r': t_r, 'x_i': x_i, 'y_i': y_i, 'C_i': C_i,
            'x_b': cat(0), 'y_b': cat(1), 't_b': cat(2), 'nx_b': cat(3), 'ny_b': cat(4)}


def make_batch(pts, obs, t_scale=1.0):
    b = dict(pts)
    b.update({k: obs[k] for k in ('x_d', 'y_d', 't_d', 'C_d')})
    b['t_scale'] = t_scale
    return {k: jnp.asarray(np.asarray(v, np.float64)) for k, v in b.items()}


# ---- bookkeeping ------------------------------------------------------------------
def load_results(name, meta):
    path = OUT_DIR / f'{name}.json'
    if path.exists():
        res = json.loads(path.read_text())
        print(f'Resuming {path.name}: {len(res["runs"])} runs already done.')
        return res
    return {**meta, 'runs': {}}


def save_results(name, res):
    (OUT_DIR / f'{name}.json').write_text(json.dumps(res, indent=2, default=float))


def pct(est, true):
    return 100.0 * (est - true) / true


def lin_predict(obs, solver, amp=C0):
    """theta = (log D, log k) -> FD concentrations at the observation points."""
    ox, oy, ot = (jnp.asarray(obs[k]) for k in ('x_d', 'y_d', 't_d'))
    C_init = gaussian_ic(solver.nx, amp)
    def predict(th):
        fr = solver(jnp.exp(th[0]), lambda C: jnp.exp(th[1]) * C, C_init)
        return interp_frames(fr, solver.dx, ox, oy, ot, solver.dt_store)
    return predict


def lin_ref_fit(obs, solver=None, D_cap=0.64, amp=C0):
    solver = SOLVE_LIN if solver is None else solver
    return ref_fit(lin_predict(obs, solver, amp), np.log([D_INIT, K_INIT]), obs['C_d'],
                   NOISE_PCT / 100 * C0, np.log([0.02, 1e-4]), np.log([0.97 * D_cap, 0.5]))


def finish(name, figs=()):
    print(f'Saved {OUT_DIR / (name + ".json")}' + ''.join(f', {f}' for f in figs))

## 4. Adaptive trainer

`build_comps` returns the four unweighted loss terms (residual, IC, BC, data) and the
residual at arbitrary points. `train_adaptive` runs the paper's Adam-then-L-BFGS schedule
with weights passed as a traced array (so changing them does not recompile), optional
learning-rate annealing of the IC/BC/data weights against the residual term, and optional
RAD resampling of the interior points. L-BFGS runs with the final weights on the final
points.

In [ ]:
from jax.flatten_util import ravel_pytree


def build_comps(physics, arch=None, pin=None):
    full = (lambda p: p) if pin is None else (lambda p: {**p, **pin})

    def resid(p, x, y, t, ts):
        q = full(p)
        f = net_fn(q, ts, arch)
        return jax.vmap(lambda a, b_, c: physics(q, a, b_, c, *point_derivs(f, a, b_, c)))(x, y, t)

    def comps(p, b):
        q = full(p)
        ts = b['t_scale']
        f = net_fn(q, ts, arch)
        res = resid(p, b['x_r'], b['y_r'], b['t_r'], ts)
        C_ic = jax.vmap(lambda x, y: f(jnp.stack([x, y, 0.0])))(b['x_i'], b['y_i'])
        dn = jax.vmap(lambda x, y, t, nx, ny:
                      jnp.dot(jax.grad(f)(jnp.stack([x, y, t]))[:2], jnp.stack([nx, ny])))(
            b['x_b'], b['y_b'], b['t_b'], b['nx_b'], b['ny_b'])
        C_d = jax.vmap(lambda x, y, t: f(jnp.stack([x, y, t])))(b['x_d'], b['y_d'], b['t_d'])
        return jnp.stack([jnp.mean(res**2), jnp.mean((C_ic - b['C_i'])**2),
                          jnp.mean(dn**2), jnp.mean((C_d - b['C_d'])**2)])

    comps.resid, comps.full, comps.arch = resid, full, arch
    return comps


W_PAPER = jnp.array([W_RES, W_IC, W_BC, W_D])


def train_adaptive(p, comps, batch, label, report, anneal=False, rad_pool=None):
    """Adam (ADAM_ITERS) + L-BFGS. Returns (params, final weights, weight history, batch).

    anneal: learning-rate annealing (Wang et al. 2021) of weights 1..3 against the
            residual term, gradients taken w.r.t. the network parameters only.
    rad_pool: callable(seed) -> (x, y, t) candidate interior points; enables RAD.
    """
    w = W_PAPER
    total = lambda q, b, w_: jnp.dot(w_, comps(q, b))
    opt = optax.adam(LR)
    state = opt.init(p)

    @jax.jit
    def chunk(q, s, b, w_):
        def body(_, carry):
            q, s, _l = carry
            l, g = jax.value_and_grad(total)(q, b, w_)
            u, s = opt.update(g, s, q)
            return optax.apply_updates(q, u), s, l
        return lax.fori_loop(0, N_CHUNK, body, (q, s, jnp.zeros((), jnp.float64)))

    @jax.jit
    def lam_hat(q, b, w_):
        def gnet(j, scale):
            g = jax.grad(lambda qq: scale * comps(qq, b)[j])(q)['net']
            return jnp.abs(ravel_pytree(g)[0])
        g_r = gnet(0, w_[0])
        return jnp.stack([jnp.max(g_r) / jnp.maximum(jnp.mean(gnet(j, 1.0)), 1e-30) for j in (1, 2, 3)])

    @jax.jit
    def pool_resid(q, x, y, t, ts):
        return comps.resid(q, x, y, t, ts)

    hist = [(0, [float(v) for v in w])]
    t0 = time.time()
    n_chunks = ADAM_ITERS // N_CHUNK
    for c in range(n_chunks):
        p, state, l = chunk(p, state, batch, w)
        step = (c + 1) * N_CHUNK
        if anneal and step % ANNEAL_EVERY == 0:
            w = w.at[1:].set((1 - ANNEAL_ALPHA) * w[1:] + ANNEAL_ALPHA * lam_hat(p, batch, w))
            if ANNEAL_CAP is not None:
                w = w.at[1:].set(jnp.clip(w[1:], W_PAPER[1:] / ANNEAL_CAP, W_PAPER[1:] * ANNEAL_CAP))
            if step % 1000 == 0:
                hist.append((step, [float(v) for v in w]))
        if rad_pool is not None and step % RAD_EVERY == 0 and step < ADAM_ITERS:
            x, y, t = rad_pool(step)
            r = np.concatenate([np.asarray(pool_resid(p, jnp.asarray(x[i:i + 20_000]), jnp.asarray(y[i:i + 20_000]),
                                                       jnp.asarray(t[i:i + 20_000]), batch['t_scale']))
                                for i in range(0, len(x), 20_000)])
            prob = np.abs(r)**RAD_K / np.mean(np.abs(r)**RAD_K) + RAD_C
            n_r = batch['x_r'].shape[0]
            idx = np.random.default_rng(step).choice(len(x), n_r, replace=False, p=prob / prob.sum())
            batch = {**batch, 'x_r': jnp.asarray(x[idx]), 'y_r': jnp.asarray(y[idx]), 't_r': jnp.asarray(t[idx])}
        if c == 0 or (c + 1) % 80 == 0 or c == n_chunks - 1:
            print(f'    [{label}] step {step:>5d}  loss {float(l):.4e}  {report(p)}  '
                  f'w = {np.array2string(np.asarray(w), precision=3)}  {time.time() - t0:.0f}s')
    p = LBFGS(fun=lambda q, b: total(q, b, w), maxiter=LBFGS_ITERS, tol=1e-9).run(p, b=batch).params
    print(f'    [{label}] after L-BFGS: {report(p)}  total {time.time() - t0:.0f}s')
    hist.append((ADAM_ITERS, [float(v) for v in w]))
    return p, w, hist, batch

## 5. The two failure cases

Same data and settings as E3b and E5 of `dopamine_PINN_extensions.ipynb` (same observation
seeds, so the `base` variant reproduces those runs up to GPU non-determinism).

In [ ]:
VMAX_TRUE, KM_TRUE = 0.0041, 0.21
OBSTACLES = [(1.5, 0.2, 0.55), (-1.4, 0.9, 0.5), (0.3, 1.6, 0.5), (-0.6, -1.6, 0.55),
             (1.2, -1.5, 0.45), (-1.7, -0.6, 0.45), (1.8, 1.7, 0.4), (-1.8, 1.9, 0.35)]
D_FREE = 0.763


def case_e3b():
    T_end, amp, sig = 500.0, 10.0, 0.02
    solver = make_fd_solver(D_cap=0.64, T_end=T_end, dt_store=0.5 if not QUICK else 1.0)
    sink = lambda Vm, Km: (lambda C: Vm * C / (Km + C))
    FR = solver(D_TRUE, sink(VMAX_TRUE, KM_TRUE), gaussian_ic(solver.nx, amp))
    obs = scattered_obs(FR, solver, 9001, noise_pct=100 * sig / C0, t_obs=(0.5, 0.9 * T_end))
    pts_kw = dict(T_end=T_end, amp=amp, t_power=2.0)
    xg = np.linspace(-L / 2, L / 2, solver.nx)
    return dict(
        name='E3b', physics=phys_mm, T_end=T_end, t_scale=T / T_end, sig=sig, obs=obs, pts_kw=pts_kw,
        truth={'logD': np.log(D_TRUE), 'logVmax': np.log(VMAX_TRUE), 'logKm': np.log(KM_TRUE)},
        init={'logD': np.log(D_INIT), 'logVmax': np.log(0.8 * VMAX_TRUE), 'logKm': np.log(1.2 * KM_TRUE)},
        pin_keys=['logD', 'logVmax', 'logKm'],
        field=dict(frames=FR, dt_store=solver.dt_store, x=xg, mask=np.ones((solver.nx,) * 2, bool),
                   times=[1.0, 10.0, 100.0, 400.0], stride=2))


def case_e5():
    nxo = 61 if QUICK else 161
    xo = np.linspace(-L / 2, L / 2, nxo)
    XO, YO = np.meshgrid(xo, xo, indexing='ij')
    mask = np.all([(XO - a)**2 + (YO - b)**2 > R**2 for a, b, R in OBSTACLES], axis=0)
    solver = make_fd_solver(nxo, D_cap=1.25, mask=mask)
    FR = solver(D_FREE, lambda C: K_TRUE * C, gaussian_ic(nxo))
    rng = np.random.default_rng(9201)
    fl = np.argwhere(mask)
    sel = fl[rng.choice(len(fl), N_OBS, replace=False)]
    ox, oy, ot = xo[sel[:, 0]], xo[sel[:, 1]], rng.uniform(*T_OBS, N_OBS)
    clean = np.asarray(interp_frames(FR, solver.dx, jnp.asarray(ox), jnp.asarray(oy), jnp.asarray(ot)))
    obs = {'x_d': ox, 'y_d': oy, 't_d': ot, 'C_clean': clean,
           'C_d': clean + rng.normal(0, NOISE_PCT / 100 * C0, N_OBS)}
    pts_kw = dict(obstacles=OBSTACLES, n_obstacle_bc=400 if QUICK else 2000)
    return dict(
        name='E5', physics=phys_linear, T_end=T, t_scale=1.0, sig=NOISE_PCT / 100 * C0, obs=obs, pts_kw=pts_kw,
        truth={'logD': np.log(D_FREE), 'logk': np.log(K_TRUE)},
        init={'logD': np.log(0.7), 'logk': np.log(K_INIT)},
        pin_keys=['logD'],
        field=dict(frames=FR, dt_store=DT_STORE, x=xo, mask=mask, times=[0.5, 2.0, 10.0, 40.0], stride=4))


def field_errors(p, case, arch):
    """Relative L2 error of the network against the FD solution at fixed times (fluid nodes)."""
    fd = case['field']
    f = net_fn(p, case['t_scale'], arch)
    xs = fd['x'][::fd['stride']]
    X, Y = np.meshgrid(xs, xs, indexing='ij')
    m = fd['mask'][::fd['stride'], ::fd['stride']]
    out = {}
    for tt in fd['times']:
        ref = np.asarray(fd['frames'][int(round(tt / fd['dt_store']))])[::fd['stride'], ::fd['stride']][m]
        pred = np.asarray(jax.vmap(lambda a, b_: f(jnp.stack([a, b_, tt])))(jnp.asarray(X[m]), jnp.asarray(Y[m])))
        out[f't={tt:g}'] = float(100 * np.linalg.norm(pred - ref) / np.linalg.norm(ref))
    return out


def run_variant(case, vname, seed, pinned=False):
    v = VARIANTS[vname]
    arch = {'ff': True, 'T_end': case['T_end']} if v['ff'] else None
    pin = {k: jnp.asarray(case['truth'][k]) for k in case['pin_keys']} if pinned else None
    comps = build_comps(case['physics'], arch=arch, pin=pin)
    pts = sample_points(**case['pts_kw'])
    batch = make_batch(pts, case['obs'], case['t_scale'])
    p0 = {'net': init_net(seed, ff=v['ff'])}
    p0.update({k: jnp.asarray(val) for k, val in case['init'].items() if not (pinned and k in case['pin_keys'])})
    rad_pool = None
    if v['rad']:
        def rad_pool(s):
            q = sample_points(n_domain=RAD_POOL * N_DOMAIN, n_initial=10, n_boundary=4, seed=10_000 + s,
                              **{k: val for k, val in case['pts_kw'].items() if k != 'n_obstacle_bc'})
            return q['x_r'], q['y_r'], q['t_r']
    rep = lambda q: ' '.join(f'{k[3:]} {float(jnp.exp(val)):.5f}' for k, val in comps.full(q).items()
                             if k.startswith('log'))
    label = f'{case["name"]}/{vname}/s{seed}' + ('/pinned' if pinned else '')
    t0 = time.time()
    p, w, hist, batch = train_adaptive(p0, comps, batch, label, rep, anneal=v['anneal'], rad_pool=rad_pool)
    q = comps.full(p)
    c = np.asarray(comps(p, batch))
    return {
        'variant': vname, 'seed': seed, 'pinned': pinned,
        'params': {k[3:]: float(jnp.exp(val)) for k, val in q.items() if k.startswith('log')},
        'err_pct': {k[3:]: float(100 * (np.exp(float(q[k]) - case['truth'][k]) - 1)) for k in case['truth']},
        'L': {'r': c[0], 'i': c[1], 'b': c[2], 'd': c[3]},
        'loss_at_paper_weights': float(np.dot(np.asarray(W_PAPER), c)),
        'rms_data_over_sigma': float(np.sqrt(c[3]) / case['sig']),
        'final_weights': [float(x) for x in w], 'weight_history': hist,
        'field_err_pct': field_errors(q, case, arch), 'minutes': (time.time() - t0) / 60}

## 6. Run the remedies on E3b and E5

In [ ]:
def run_case(make_case, flag):
    if not flag:
        return None
    case = make_case()
    name = f'R_{case["name"]}'
    path = OUT_DIR / f'{name}.json'
    res = json.loads(path.read_text()) if path.exists() else {'runs': {}}
    jobs = [(v, s, False) for v in VARIANTS for s in NET_SEEDS] + [(v, NET_SEEDS[0], True) for v in PINNED_VARIANTS]
    for vname, seed, pinned in jobs:
        key = f'{vname}_s{seed}' + ('_pinned' if pinned else '')
        if key in res['runs']:
            continue
        print('=' * 78); print(f'  {name}: {key}'); print('=' * 78)
        res['runs'][key] = run_variant(case, vname, seed, pinned)
        path.write_text(json.dumps(res, indent=2, default=float))
        r = res['runs'][key]
        print(f'  -> {key}: errors ' + ', '.join(f'{k} {v:+.2f}%' for k, v in r['err_pct'].items()
                                               if not (pinned and 'log' + k in case['pin_keys']))
              + f' | field err ' + ', '.join(f'{k} {v:.1f}%' for k, v in r['field_err_pct'].items())
              + f' | loss@paper w {r["loss_at_paper_weights"]:.3e} | {r["minutes"]:.1f} min')
    return case, res


def summarize(case, res):
    R = res['runs']
    keys = list(case['truth'])
    print(f'\n  {case["name"]}: signed % errors, mean over seeds (and field error vs FD, mean over times)')
    print('  ' + f'{"variant":<8}' + ''.join(f'{k[3:]:>10}' for k in keys) + f'{"field":>9}{"RMS/sig":>9}')
    for v in VARIANTS:
        rs = [R[f'{v}_s{s}'] for s in NET_SEEDS if f'{v}_s{s}' in R]
        if not rs:
            continue
        errs = [np.mean([r['err_pct'][k[3:]] for r in rs]) for k in keys]
        fe = np.mean([np.mean(list(r['field_err_pct'].values())) for r in rs])
        rms = np.mean([r['rms_data_over_sigma'] for r in rs])
        print('  ' + f'{v:<8}' + ''.join(f'{e:>+9.1f}%' for e in errs) + f'{fe:>8.1f}%{rms:>9.3f}')
    for v in PINNED_VARIANTS:
        kp, kl = f'{v}_s{NET_SEEDS[0]}_pinned', f'{v}_s{NET_SEEDS[0]}'
        if kp in R and kl in R:
            lp, ll = R[kp]['loss_at_paper_weights'], R[kl]['loss_at_paper_weights']
            verdict = 'truth has the LOWER loss (objective OK)' if lp < ll else 'truth has the HIGHER loss (objective biased)'
            print(f'  pinned check [{v}]: loss at paper weights, truth {lp:.3e} vs learned {ll:.3e} -> {verdict}')

    vs = [v for v in VARIANTS if f'{v}_s{NET_SEEDS[0]}' in R]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    xi = np.arange(len(vs))
    for j, k in enumerate(keys):
        m = [np.mean([R[f'{v}_s{s}']['err_pct'][k[3:]] for s in NET_SEEDS if f'{v}_s{s}' in R]) for v in vs]
        ax[0].bar(xi + (j - (len(keys) - 1) / 2) * 0.25, m, 0.25, label=k[3:])
    ax[0].axhline(0, color='k', lw=0.8); ax[0].set_xticks(xi); ax[0].set_xticklabels(vs)
    ax[0].set_ylabel('signed error (%)'); ax[0].set_title(f'{case["name"]}: parameter errors'); ax[0].legend()
    ax[0].grid(alpha=0.3, axis='y')
    for v in vs:
        r = R[f'{v}_s{NET_SEEDS[0]}']
        ax[1].semilogy(range(len(r['field_err_pct'])), list(r['field_err_pct'].values()), 'o-', label=v)
    ax[1].set_xticks(range(len(case['field']['times'])))
    ax[1].set_xticklabels([f't = {t:g} ms' for t in case['field']['times']])
    ax[1].set_ylabel('network vs FD, rel. L2 (%)'); ax[1].set_title('field accuracy (seed 1)'); ax[1].legend()
    ax[1].grid(alpha=0.3, which='both')
    fig.tight_layout(); fig.savefig(OUT_DIR / f'R_{case["name"]}.png', dpi=150); plt.show()


for make_case, flag in ((case_e3b, RUN_E3B), (case_e5, RUN_E5)):
    out = run_case(make_case, flag)
    if out is not None:
        summarize(*out)

## 7. Optional: the chosen variant on five E1 realizations

Set `E1_VARIANT` after looking at the results above. Baseline values from E1 of
`dopamine_PINN_extensions.ipynb` (network seed 1234) are printed alongside.

In [ ]:
E1_BASE = {7001: (-10.21, -1.85), 7002: (-8.86, 11.21), 7003: (-4.05, -13.24),
           7004: (-7.37, -4.48), 7005: (1.15, 1.35)}           # (PINN D err %, k err %)
E1_REF = {7001: (-8.05, -2.10), 7002: (-4.54, 8.95), 7003: (0.31, -2.34),
          7004: (-4.71, -10.83), 7005: (3.59, -3.34)}          # reference estimator
if not RUN_E1_SUBSET:
    print('RUN_E1_SUBSET = False, skipping.')
else:
    SOLVE_LIN_ = make_fd_solver(D_cap=0.64)
    FR_LIN_ = SOLVE_LIN_(D_TRUE, lambda C: K_TRUE * C, gaussian_ic(SOLVE_LIN_.nx))
    path = OUT_DIR / f'R_E1_subset_{E1_VARIANT}.json'
    res = json.loads(path.read_text()) if path.exists() else {'variant': E1_VARIANT, 'runs': {}}
    for s in list(E1_BASE)[:(1 if QUICK else 5)]:
        if str(s) in res['runs']:
            continue
        case = dict(name=f'E1_{s}', physics=phys_linear, T_end=T, t_scale=1.0, sig=0.02,
                    obs=scattered_obs(FR_LIN_, SOLVE_LIN_, s), pts_kw={},
                    truth={'logD': np.log(D_TRUE), 'logk': np.log(K_TRUE)},
                    init={'logD': np.log(D_INIT), 'logk': np.log(K_INIT)}, pin_keys=['logD'],
                    field=dict(frames=FR_LIN_, dt_store=DT_STORE, x=np.linspace(-L / 2, L / 2, SOLVE_LIN_.nx),
                               mask=np.ones((SOLVE_LIN_.nx,) * 2, bool), times=[1.0, 5.0, 40.0], stride=2))
        res['runs'][str(s)] = run_variant(case, E1_VARIANT, SEED)
        path.write_text(json.dumps(res, indent=2, default=float))
    print(f'\n  {"seed":<6}{"D: base":>9}{E1_VARIANT:>9}{"ref":>8} |{"k: base":>9}{E1_VARIANT:>9}{"ref":>8}')
    for s, r in res['runs'].items():
        b, f = E1_BASE[int(s)], E1_REF[int(s)]
        print(f'  {s:<6}{b[0]:>+8.1f}%{r["err_pct"]["D"]:>+8.1f}%{f[0]:>+7.1f}% |'
              f'{b[1]:>+8.1f}%{r["err_pct"]["k"]:>+8.1f}%{f[1]:>+7.1f}%')

## Download everything

In [ ]:
import shutil
zip_path = shutil.make_archive(str(OUT_DIR.parent / 'dopamine_PINN_remedies_results'), 'zip', OUT_DIR)
print(f'Zipped results: {zip_path}')
try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except ImportError:
    print('(Not on Colab - results stay in', OUT_DIR, ')')